# Tratamento gold — Cota (Parquet)

Pega a série diária de cota já limpa pela silver (`cota_diaria.csv`) e
grava em Parquet, com cada coluna no tipo certo — mesma lógica do
`gerar_parquet_chuva.ipynb`.

Entrada (silver): `data/silver/ana/cota/cota_diaria.csv`
Saída (gold):     `data/gold/ana/cota/cota_diaria.parquet`

In [ ]:
from pathlib import Path

import pandas as pd


def encontrar_raiz_do_projeto(marcador: str = "pyproject.toml") -> Path:
    atual = Path.cwd().resolve()
    for pasta in [atual, *atual.parents]:
        if (pasta / marcador).exists():
            return pasta
    raise FileNotFoundError(
        f"Não encontrei '{marcador}' subindo a partir de {atual}. "
        "Este notebook está dentro da pasta do projeto?"
    )


RAIZ = encontrar_raiz_do_projeto()

ENTRADA_CSV = RAIZ / "data" / "silver" / "ana" / "cota" / "cota_diaria.csv"
SAIDA_PARQUET = RAIZ / "data" / "gold" / "ana" / "cota" / "cota_diaria.parquet"

print("Raiz do projeto:", RAIZ)
print("Lendo de:", ENTRADA_CSV)
print("Vai salvar em:", SAIDA_PARQUET)

## 1. Ler o CSV da silver

In [ ]:
silver = pd.read_csv(ENTRADA_CSV)
print(f"{len(silver)} linha(s) lida(s) da silver.")
silver.dtypes

## 2. Aplicar o schema final (tipo de cada coluna)

In [ ]:
TIPOS_FINAIS = {
    "codigoestacao": "Int64",
    "cota_cm": "float32",
    "status": "Int64",
    "nivel_consistencia": "Int64",
    "tipo_medicao": "Int64",
}

gold = silver.copy()
gold["data"] = pd.to_datetime(gold["data"]).dt.date

for coluna, tipo in TIPOS_FINAIS.items():
    gold[coluna] = pd.to_numeric(gold[coluna], errors="coerce").astype(tipo)

gold.dtypes

## 3. Traduzir `nivel_consistencia` pra texto

Mesma fonte/ressalva do notebook de chuva: 1 = Bruto, 2 = Consistido
(documentado pela ANA em serviço parecido, não exatamente este endpoint
REST). `status` e `tipo_medicao` continuam numéricos — sem documentação
oficial confirmada pra esse endpoint.

In [ ]:
NIVEL_CONSISTENCIA_LABELS = {
    1: "Bruto",
    2: "Consistido",
}

gold["nivel_consistencia_desc"] = (
    gold["nivel_consistencia"]
    .map(NIVEL_CONSISTENCIA_LABELS)
    .fillna("Desconhecido (código " + gold["nivel_consistencia"].astype(str) + ")")
)

gold["nivel_consistencia_desc"].value_counts()

## 4. Trazer o nome da estação (`nm_estacao`)

Faz um `merge` com a tabela de estações (silver) pra trazer o nome de cada
estação direto pra dentro da tabela de cota.

In [ ]:
ESTACOES_CSV = RAIZ / "data" / "silver" / "ana" / "estacoes" / "estacoes.csv"

estacoes = pd.read_csv(ESTACOES_CSV)[["codigo", "nome"]].rename(
    columns={"codigo": "codigoestacao", "nome": "nm_estacao"}
)

gold = gold.merge(estacoes, on="codigoestacao", how="left")

sem_nome = gold["nm_estacao"].isna().sum()
print(f"Linhas sem nome de estação encontrado: {sem_nome} de {len(gold)}")
gold["nm_estacao"] = gold["nm_estacao"].fillna("Desconhecida")

## 5. Padronizar nomes de colunas (prefixos)

Adiciona colunas com o padrão `cod_`/`nm_`/`dt_`/`vl_`/`ds_`, **sem apagar**
as colunas originais.

- `ds_status` e `ds_tipo_medicao` ficam como "Não documentado (código X)"
  — mesma ressalva da chuva: sem fonte oficial confirmada pra esses dois
  códigos nesse endpoint. Só `ds_nivel_consistencia` tem descrição real.

In [ ]:
gold["cod_estacao"] = gold["codigoestacao"]
gold["dt_medicao"] = gold["data"]
gold["vl_cota_cm"] = gold["cota_cm"]

gold["cod_nivel_consistencia"] = gold["nivel_consistencia"]
gold["ds_nivel_consistencia"] = gold["nivel_consistencia_desc"]

gold["cod_status"] = gold["status"]
gold["ds_status"] = gold["status"].apply(
    lambda v: "Sem código" if pd.isna(v) else f"Não documentado (código {v})"
)

gold["cod_tipo_medicao"] = gold["tipo_medicao"]
gold["ds_tipo_medicao"] = gold["tipo_medicao"].apply(
    lambda v: "Sem código" if pd.isna(v) else f"Não documentado (código {v})"
)

gold.columns.tolist()

## 6. Conferência rápida

In [ ]:
print("Linhas:", len(gold))
print("Estações distintas:", gold["codigoestacao"].nunique())
print("Nulos por coluna:")
print(gold.isna().sum())
gold.head()

## 7. Salvar em Parquet (gold)

In [ ]:
SAIDA_PARQUET.parent.mkdir(parents=True, exist_ok=True)
gold.to_parquet(SAIDA_PARQUET, index=False, engine="pyarrow")

tamanho_csv_mb = ENTRADA_CSV.stat().st_size / 1e6
tamanho_parquet_mb = SAIDA_PARQUET.stat().st_size / 1e6
print(f"Salvo em {SAIDA_PARQUET}")
print(f"Tamanho: {tamanho_csv_mb:.1f} MB (csv) -> {tamanho_parquet_mb:.1f} MB (parquet)")